# 06 · 복원 경로 선택 가능성 점검 (validation only)

목표: 피드백 조건(COST2100 Indoor, CR=1/4)을 고정하고 CsiNet·CRNet·TransNet 중 샘플마다 더 적합한 경로가 있는지 확인합니다. 먼저 **oracle 상한**을 계산합니다. Oracle은 정답 CSI를 알아야 선택할 수 있으므로 배포 정책이 아니며, sLLM 학습 결과도 아닙니다. oracle 이득이 거의 없으면 라우팅 연구를 진행하지 않습니다.

이 노트북은 validation split만 읽고 test split은 읽지 않습니다. 체크포인트 학습 recipe/epoch가 다르므로 결과는 feasibility 판단용이며 모델 성능 순위 주장이 아닙니다.


In [ ]:
from pathlib import Path
import sys, time, json, csv
import numpy as np
import torch
from torch.utils.data import DataLoader
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/CSI_SLLM_RESEARCH')
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT_OVERRIDE = None  # 선택사항: Path 문자열 지정
if PROJECT_ROOT_OVERRIDE: PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE)
sys.path.insert(0, str(PROJECT_ROOT))
from src.train_baseline import build_model
from src.cost2100 import COST2100Dataset
from src.metrics import sample_nmse_linear
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
VAL_PATH = PROJECT_ROOT / 'Data' / 'DATA_Hvalin.mat'
VAL_BATCH_SIZE = 128
print('Project:', PROJECT_ROOT, '| device:', DEVICE, '| validation:', VAL_PATH)


## 체크포인트 선택

기본값은 실제 존재하는 CsiNet, CRNet 100 epoch, TransNet 400 epoch checkpoint입니다. 각 모델의 학습이 동일 조건은 아니므로 이 단계에서는 전체 모델을 후보 경로로 놓을 수 있는지의 대략적인 가능성만 확인합니다. 파일이 없거나 재학습 후 다른 checkpoint를 쓰려면 경로를 바꾸세요.


In [ ]:
CHECKPOINTS = {
    'CsiNet': PROJECT_ROOT / 'runs' / 'csinet_indoor_cr4' / 'best.pt',
    'CRNet': PROJECT_ROOT / 'runs' / 'crnet_indoor_cr4_100ep' / 'best.pt',
    'TransNet': PROJECT_ROOT / 'runs' / 'transnet_indoor_cr4_400ep' / 'best.pt',
}
missing = [str(p) for p in CHECKPOINTS.values() if not p.is_file()]
if missing: raise FileNotFoundError(', '.join(missing))
val_ds = COST2100Dataset(VAL_PATH)
val_loader = DataLoader(val_ds, batch_size=VAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=(DEVICE.type == 'cuda'))
print('Validation samples:', len(val_ds))


## 모델별 품질과 batch-1 latency 측정

각 모델을 validation 전체에 한 번씩 실행해 샘플별 NMSE를 저장합니다. latency는 batch 1의 warm-up 후 반복 측정 median입니다. 로컬/GPU, dtype, clock 상태에 따라 바뀌므로 여기서 잰 숫자는 동일 세션의 비교용입니다.


In [ ]:
def sync():
    if DEVICE.type == 'cuda': torch.cuda.synchronize()

def measure_latency_ms(model, sample, warmup=20, repeats=100):
    model.eval()
    with torch.inference_mode():
        for _ in range(warmup): _ = model(sample)
        sync()
        times=[]
        for _ in range(repeats):
            sync(); t0=time.perf_counter(); _=model(sample); sync()
            times.append((time.perf_counter()-t0)*1000)
    return float(np.median(times)), float(np.percentile(times,95))

models, nmse_by_model, latency = {}, {}, {}
for name, ckpt_path in CHECKPOINTS.items():
    model = build_model(name.lower(), PROJECT_ROOT).to(DEVICE)
    state = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    model.load_state_dict(state['model'])
    model.eval(); models[name] = model
    ratios=[]
    with torch.inference_mode():
        for x in val_loader:
            x=x.to(DEVICE, non_blocking=True)
            ratios.append(sample_nmse_linear(x, model(x)).cpu().numpy())
    nmse_by_model[name]=np.concatenate(ratios)
    sample=val_ds[0:1].to(DEVICE)
    latency[name]=measure_latency_ms(model, sample)
    mean_db=10*np.log10(max(float(nmse_by_model[name].mean()),1e-12))
    print(f'{name:<10} val_NMSE={mean_db:.3f} dB | latency median/p95={latency[name][0]:.3f}/{latency[name][1]:.3f} ms')

out_dir=PROJECT_ROOT/'runs'/'router_feasibility_indoor_cr4'; out_dir.mkdir(parents=True,exist_ok=True)
sample_csv=out_dir/'validation_sample_nmse.csv'
with sample_csv.open('w',newline='',encoding='utf-8') as f:
    w=csv.writer(f); w.writerow(['sample_index', *[f'{n}_nmse_linear' for n in nmse_by_model]])
    for i in range(len(val_ds)): w.writerow([i, *[float(nmse_by_model[n][i]) for n in nmse_by_model]])
latency_json={n:{'median_ms':latency[n][0],'p95_ms':latency[n][1]} for n in latency}
(out_dir/'latency.json').write_text(json.dumps(latency_json,indent=2),encoding='utf-8')
print('Saved:', sample_csv)


## Oracle 상한: 라우팅을 해볼 가치가 있는가?

oracle은 각 샘플의 정답 NMSE를 보고 후보 중 가장 낮은 NMSE의 모델을 고릅니다. 실전에서 정답은 모르기 때문에 달성 가능한 시스템은 아니며, 입력만으로 가능한 정책의 낙관적 상한입니다. 전체 평균 기준 가장 좋은 고정 모델과 비교합니다. latency 제한이 있으면 그 제한 안의 모델만 oracle 선택에 허용합니다.


In [ ]:
names=list(nmse_by_model)
matrix=np.stack([nmse_by_model[n] for n in names],axis=1)
fixed_idx=int(matrix.mean(axis=0).argmin())
fixed_nmse=float(matrix[:,fixed_idx].mean())
oracle_nmse=float(matrix.min(axis=1).mean())
print(f'Best fixed model: {names[fixed_idx]} | {10*np.log10(fixed_nmse):.3f} dB | {latency[names[fixed_idx]][0]:.3f} ms median')
print(f'Unconstrained oracle: {10*np.log10(oracle_nmse):.3f} dB | impossible upper bound')
print(f'Oracle gain over best fixed: {10*np.log10(fixed_nmse/oracle_nmse):.3f} dB')

budgets=sorted(set(round(latency[n][0],6) for n in names))
rows=[]
for budget in budgets:
    allowed=[j for j,n in enumerate(names) if latency[n][0] <= budget+1e-9]
    best_per_sample=matrix[:,allowed].min(axis=1)
    rows.append((budget,[names[j] for j in allowed],10*np.log10(best_per_sample.mean())))
print('Latency budget | allowed paths | oracle NMSE dB')
for budget,allowed,db in rows: print(f'{budget:.3f} ms | {allowed} | {db:.3f} dB')
summary={'fixed_best_model':names[fixed_idx],'fixed_best_nmse_db':float(10*np.log10(fixed_nmse)),'unconstrained_oracle_nmse_db':float(10*np.log10(oracle_nmse)),'oracle_gain_db':float(10*np.log10(fixed_nmse/oracle_nmse)),'latency':latency_json,'latency_budget_oracles':[{'budget_ms':b,'allowed_models':a,'nmse_db':float(db)} for b,a,db in rows],'interpretation':'Oracle uses ground-truth labels and is an unattainable upper bound; do not report as a deployed method.'}
(out_dir/'oracle_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print('Saved:', out_dir/'oracle_summary.json')


## 판단 기준

- oracle가 best fixed보다 거의 좋아지지 않으면, 입력별 모델 선택만으로 얻을 수 있는 여지가 작으므로 이 방향을 다시 검토합니다.
- oracle 이득이 보여도 실제 정책이 이득을 재현한다는 뜻은 아닙니다. 다음 단계에서 CSI 입력만 보는 규칙/LUT와 작은 MLP를 먼저 학습하고, held-out validation subset/seed에서 일반화와 정책 오버헤드를 평가해야 합니다.
- 현재 세 체크포인트는 학습 recipe/epoch가 다릅니다. 본 결과는 라우팅 행동 공간의 예비 점검일 뿐, 모델 자체의 공정 순위 비교나 최종 논문 결과가 아닙니다.
- test split은 이 노트북에서 쓰지 않습니다.
